# Transformer Template

# ├── 1. Configuration
# ├── 2. Tokenizer
# ├── 3. Dataset
# ├── 4. Positional Encoding
# ├── 5. Self-Attention
# ├── 6. Multi-Head Attention
# ├── 7. Feed Forward
# ├── 8. Transformer Block
# ├── 9. Full Transformer
# ├── 10. Training
# ├── 11. Evaluation
# └── 12. Generation

# 1. Import the Libraries

In [42]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# Configuration

In [43]:
config = {
    "vocab_size": 10000,
    "embed_dim": 128,
    "num_heads": 4,
    "num_layers": 4,
    "max_seq_len": 128,
    "dropout": 0.1,
    "batch_size": 32,
    "learning_rate": 3e-4,
    "epochs": 10
}

print(config)

{'vocab_size': 10000, 'embed_dim': 128, 'num_heads': 4, 'num_layers': 4, 'max_seq_len': 128, 'dropout': 0.1, 'batch_size': 32, 'learning_rate': 0.0003, 'epochs': 10}


# Device

In [44]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using:", device)

Using: cpu


# Configuring Variables

In [45]:
vocab_size = config["vocab_size"]
embed_dim = config["embed_dim"]
num_heads = config["num_heads"]
num_layers = config["num_layers"]
max_seq_len = config["max_seq_len"]
dropout = config["dropout"]
batch_size = config["batch_size"]
learning_rate = config["learning_rate"]
epochs = config["epochs"]

# 2. Tokenizer

In [46]:
class SimpleTokenizer:

    def __init__(self, text):

        # Create vocabulary
        tokens = text.lower().split()

        vocab = sorted(set(tokens))

        # Special token
        self.pad_token = "<PAD>"

        vocab = [self.pad_token] + vocab

        # Word → ID
        self.stoi = {
            word: i
            for i, word in enumerate(vocab)
        }

        # ID → Word
        self.itos = {
            i: word
            for word, i in self.stoi.items()
        }

    def encode(self, text):

        tokens = text.lower().split()

        return [
            self.stoi[token]
            for token in tokens
        ]

    def decode(self, ids):

        return " ".join(
            self.itos[i]
            for i in ids
        )

    @property
    def vocab_size(self):

        return len(self.stoi)

**test the tokenizer**

In [47]:
text = """
i love cats and dogs
cats are cute
dogs are friendly
i love animals
animals are beautiful
"""

In [48]:
tokenizer = SimpleTokenizer(text)

In [49]:
print("Vocabulary size:", tokenizer.vocab_size)

print(tokenizer.stoi)

Vocabulary size: 11
{'<PAD>': 0, 'and': 1, 'animals': 2, 'are': 3, 'beautiful': 4, 'cats': 5, 'cute': 6, 'dogs': 7, 'friendly': 8, 'i': 9, 'love': 10}


**Encode**

In [50]:
encoded = tokenizer.encode(text)
print(encoded)

[9, 10, 5, 1, 7, 5, 3, 6, 7, 3, 8, 9, 10, 2, 2, 3, 4]


**Decode**

In [51]:
decoded = tokenizer.decode(encoded)
print(decoded)

i love cats and dogs cats are cute dogs are friendly i love animals animals are beautiful


# 3. Dataset

In [52]:
class LanguageModelDataset(Dataset):

    def __init__(self, token_ids, seq_len):

        self.token_ids = token_ids
        self.seq_len = seq_len

    def __len__(self):

        return len(self.token_ids) - self.seq_len

    def __getitem__(self, index):

        input_ids = self.token_ids[
            index:index + self.seq_len
        ]

        target_ids = self.token_ids[
            index + 1:index + self.seq_len + 1
        ]

        return (
            torch.tensor(input_ids, dtype=torch.long),
            torch.tensor(target_ids, dtype=torch.long)
        )

**convert our text to ids**

In [53]:
token_ids = tokenizer.encode(text)
print(token_ids)

[9, 10, 5, 1, 7, 5, 3, 6, 7, 3, 8, 9, 10, 2, 2, 3, 4]


In [54]:
dataset = LanguageModelDataset(
    token_ids,
    seq_len=4
)

**eg:**

In [55]:
X, y = dataset[0]

print("Input IDs :", X)
print("Target IDs:", y)

Input IDs : tensor([ 9, 10,  5,  1])
Target IDs: tensor([10,  5,  1,  7])


**decode them**

In [56]:
print(
    "Input :",
    tokenizer.decode(X.tolist())
)

print(
    "Target:",
    tokenizer.decode(y.tolist())
)

Input : i love cats and
Target: love cats and dogs


**Create DataLoader**

In [57]:
train_loader = DataLoader(
    dataset,
    batch_size=config["batch_size"],
    shuffle=True
)

In [58]:
X_batch, y_batch = next(iter(train_loader))

print("X batch:", X_batch.shape)
print("Y batch:", y_batch.shape)

X batch: torch.Size([13, 4])
Y batch: torch.Size([13, 4])


# 4. Embedding & Positional Encoding

**Create the embedding layer**

In [59]:
class TokenEmbedding(nn.Module):

    def __init__(self, vocab_size, embed_dim):

        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embed_dim
        )

    def forward(self, x):

        return self.embedding(x)

In [60]:
embedding = TokenEmbedding(
    vocab_size=tokenizer.vocab_size,
    embed_dim=config["embed_dim"]
)

**Positional Encoding**

In [61]:
class PositionalEmbedding(nn.Module):

    def __init__(self, max_seq_len, embed_dim):

        super().__init__()

        self.position_embedding = nn.Embedding(
            max_seq_len,
            embed_dim
        )

    def forward(self, x):

        batch_size, seq_len, embed_dim = x.shape

        positions = torch.arange(
            seq_len,
            device=x.device
        )

        position_vectors = self.position_embedding(
            positions
        )

        return x + position_vectors

**combine embedding + positional encoding**

In [62]:
position_embedding = PositionalEmbedding(
    max_seq_len=config["max_seq_len"],
    embed_dim=config["embed_dim"]
)

In [63]:
embedded = embedding(X_batch)

embedded = position_embedding(
    embedded
)

print(embedded.shape)

torch.Size([13, 4, 128])


# The Embedding Layer

In [64]:
class InputEmbedding(nn.Module):

    def __init__(
        self,
        vocab_size,
        embed_dim,
        max_seq_len
    ):

        super().__init__()

        self.token_embedding = nn.Embedding(
            vocab_size,
            embed_dim
        )

        self.position_embedding = nn.Embedding(
            max_seq_len,
            embed_dim
        )

    def forward(self, x):

        batch_size, seq_len = x.shape

        positions = torch.arange(
            seq_len,
            device=x.device
        )

        token_vectors = self.token_embedding(x)

        position_vectors = self.position_embedding(
            positions
        )

        return token_vectors + position_vectors

In [65]:
input_embedding = InputEmbedding(
    vocab_size=tokenizer.vocab_size,
    embed_dim=config["embed_dim"],
    max_seq_len=config["max_seq_len"]
)

In [66]:
X_batch, y_batch = next(iter(train_loader))

x = input_embedding(X_batch)

print(x.shape)

torch.Size([13, 4, 128])


# 5. Causal Multi-Head Self-Attention

In [67]:
class MultiHeadAttention(nn.Module):

    def __init__(self, embed_dim, num_heads, dropout=0.1):
        super().__init__()

        assert embed_dim % num_heads == 0

        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads

        self.qkv = nn.Linear(embed_dim, 3 * embed_dim)
        self.out_proj = nn.Linear(embed_dim, embed_dim)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):

        B, T, C = x.shape

        qkv = self.qkv(x)

        q, k, v = qkv.chunk(3, dim=-1)

        q = q.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)

        scores = (q @ k.transpose(-2, -1)) / (self.head_dim ** 0.5)

        # Causal mask
        mask = torch.tril(
            torch.ones(T, T, device=x.device)
        )

        scores = scores.masked_fill(
            mask == 0,
            float("-inf")
        )

        attention = F.softmax(scores, dim=-1)

        attention = self.dropout(attention)

        out = attention @ v

        out = out.transpose(1, 2).contiguous()

        out = out.view(B, T, C)

        return self.out_proj(out)

# 6. Feed Forward Network

In [68]:
class FeedForward(nn.Module):

    def __init__(self, embed_dim, dropout=0.1):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(embed_dim, 4 * embed_dim),
            nn.GELU(),
            nn.Linear(4 * embed_dim, embed_dim),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.network(x)

# 7. Transformer Block

In [69]:
class TransformerBlock(nn.Module):

    def __init__(
        self,
        embed_dim,
        num_heads,
        dropout=0.1
    ):
        super().__init__()

        self.ln1 = nn.LayerNorm(embed_dim)

        self.attention = MultiHeadAttention(
            embed_dim,
            num_heads,
            dropout
        )

        self.ln2 = nn.LayerNorm(embed_dim)

        self.ffn = FeedForward(
            embed_dim,
            dropout
        )

    def forward(self, x):

        # Attention + residual
        x = x + self.attention(
            self.ln1(x)
        )

        # FFN + residual
        x = x + self.ffn(
            self.ln2(x)
        )

        return x

In [71]:
class TinyGPT(nn.Module):

    def __init__(
        self,
        vocab_size,
        embed_dim,
        num_heads,
        num_layers,
        max_seq_len,
        dropout
    ):

        super().__init__()

        self.max_seq_len = max_seq_len

        self.input_embedding = InputEmbedding(
            vocab_size,
            embed_dim,
            max_seq_len
        )

        self.transformer_blocks = nn.Sequential(
            *[TransformerBlock(
                embed_dim,
                num_heads,
                dropout
            ) for _ in range(num_layers)]
        )

        self.ln = nn.LayerNorm(embed_dim)

        self.linear = nn.Linear(embed_dim, vocab_size)

    def forward(self, x):

        x = self.input_embedding(x)

        x = self.transformer_blocks(x)

        x = self.ln(x)

        logits = self.linear(x)

        return logits

model = TinyGPT(
    vocab_size=tokenizer.vocab_size,
    embed_dim=config["embed_dim"],
    num_heads=config["num_heads"],
    num_layers=config["num_layers"],
    max_seq_len=config["max_seq_len"],
    dropout=config["dropout"]
).to(device)

# Training

In [72]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=config["learning_rate"]
)

for epoch in range(config["epochs"]):

    model.train()

    total_loss = 0

    for X_batch, y_batch in train_loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        logits = model(X_batch)

        B, T, V = logits.shape

        loss = criterion(
            logits.view(B * T, V),
            y_batch.view(B * T)
        )

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    print(
        f"Epoch {epoch + 1}/{config['epochs']} "
        f"| Loss: {avg_loss:.4f}"
    )

Epoch 1/10 | Loss: 2.5085
Epoch 2/10 | Loss: 2.2157
Epoch 3/10 | Loss: 1.9481
Epoch 4/10 | Loss: 1.7529
Epoch 5/10 | Loss: 1.5617
Epoch 6/10 | Loss: 1.3981
Epoch 7/10 | Loss: 1.2821
Epoch 8/10 | Loss: 1.1449
Epoch 9/10 | Loss: 0.9985
Epoch 10/10 | Loss: 0.9330


**sample generation**

In [73]:
def generate(
    model,
    tokenizer,
    prompt,
    max_new_tokens=20,
    temperature=1.0
):

    model.eval()

    tokens = tokenizer.encode(prompt)

    x = torch.tensor(
        [tokens],
        dtype=torch.long,
        device=device
    )

    for _ in range(max_new_tokens):

        x_cond = x[:, -model.max_seq_len:]

        with torch.no_grad():

            logits = model(x_cond)

        logits = logits[:, -1, :]

        logits = logits / temperature

        probs = F.softmax(
            logits,
            dim=-1
        )

        next_token = torch.multinomial(
            probs,
            num_samples=1
        )

        x = torch.cat(
            [x, next_token],
            dim=1
        )

    return tokenizer.decode(
        x[0].tolist()
    )

In [79]:
print(
    generate(
        model,
        tokenizer,
        "i love",
        max_new_tokens=10
    )
)

i love cute dogs love cats are friendly i friendly i beautiful
